# Mastering Python Dictionaries for AI Engineering

## Learning Goal
- **What this notebook teaches**: Python dicts from fundamentals to production patterns — especially column-oriented dicts, nested structures, and the dict-as-data-bus pattern.
- **Background & links**: Python data model docs: https://docs.python.org/3/library/stdtypes.html#mapping-types-dict
- **Why it matters in AI engineering**: Dicts are the universal in-memory container. Every API response you parse, every model config you load, every RAG metadata record, and every intermediate data structure in a pipeline is a dict. They are also the natural bridge between JSON, Pandas, and NumPy.

## Mental Model

```
                   ┌─────────────────────────────────────────┐
                   │           Python Dict                    │
                   │  { key: value, key: value, ... }        │
                   │                                         │
  JSON string ────▶│  row-oriented   →  list of dicts        │
  (parsed)         │  col-oriented   →  dict of lists   ────▶  DataFrame
                   │  nested         →  dict of dicts        │
                   │  counter        →  dict of counts       │
                   └─────────────────────────────────────────┘
```

A dict is a hash map: each key hashes to a memory address where the value lives.
Lookup `d[key]` is O(1) regardless of how large the dict is.

**Two orientations matter in data work:**
- **Row-oriented** (`[{"name":"Alice","score":85}, ...]`) — natural JSON format, easy to read
- **Column-oriented** (`{"name":["Alice",...], "score":[85,...]}`) — Pandas-ready, memory-efficient

## Background Explanation

### Core operations

| Operation | Syntax | Notes |
|-----------|--------|-------|
| Create | `d = {"k": v}` or `dict(k=v)` | |
| Access | `d["k"]` | Raises `KeyError` if missing |
| Safe access | `d.get("k", default)` | Returns default if missing |
| Insert/update | `d["k"] = v` | |
| Delete | `del d["k"]` or `d.pop("k")` | |
| Merge (3.9+) | `d1 \| d2` | d2 values win on conflict |
| Keys / values | `d.keys()`, `d.values()`, `d.items()` | Views, not copies |
| Membership | `"k" in d` | O(1) hash lookup |

### Dict comprehension
```python
{k: transform(v) for k, v in d.items() if condition(k)}
```

### Important: dicts preserve insertion order (Python 3.7+)

In [ ]:
# Quick orientation
d = {"model": "arabic-bert", "dim": 768, "layers": 12}
print(d["model"])             # arabic-bert
print(d.get("lr", 1e-4))     # 0.0001 (safe default)
print(list(d.keys()))         # ['model', 'dim', 'layers']
print(list(d.items()))        # [('model', 'arabic-bert'), ...]
print("dim" in d)             # True

## Minimal Working Example — Dict as a Data Pipeline Stage

In [ ]:
# Raw model config dict
config = {
    "model_name": "AraGPT2-base",
    "training": {
        "lr": 2e-5,
        "epochs": 3,
        "batch_size": 16,
    },
    "data": {
        "train_path": "data/train.jsonl",
        "val_path":   "data/val.jsonl",
    }
}

# Access nested keys safely
lr    = config["training"]["lr"]
model = config.get("model_name", "unknown")
print(f"Training {model} at lr={lr}")

# Dict comprehension — filter only numeric config values
numeric_cfg = {k: v for k, v in config["training"].items() if isinstance(v, (int, float))}
print("Numeric training params:", numeric_cfg)

## Exercise 1 — Build a Token Counter and Cost Estimator

**Goal**: Parse a list of LLM call records (list of dicts) and compute per-model token usage and estimated cost.

**Real context**: In production, you track token usage per model to estimate monthly API bills.

In [ ]:
# Cost per 1M tokens (USD) — fictional but realistic structure
COST_PER_M = {
    "claude-sonnet-4-5": {"input": 3.00, "output": 15.00},
    "gpt-4o":             {"input": 2.50, "output": 10.00},
    "arabic-llm":         {"input": 1.00, "output": 4.00},
}

call_log = [
    {"model": "claude-sonnet-4-5", "input_tokens": 512,  "output_tokens": 128},
    {"model": "claude-sonnet-4-5", "input_tokens": 1024, "output_tokens": 256},
    {"model": "gpt-4o",            "input_tokens": 800,  "output_tokens": 200},
    {"model": "arabic-llm",        "input_tokens": 400,  "output_tokens": 100},
    {"model": "gpt-4o",            "input_tokens": 300,  "output_tokens":  80},
]

def estimate_costs(log: list, costs: dict) -> dict:
    """
    Return a dict: { model_name: {"total_input_tokens": int,
                                   "total_output_tokens": int,
                                   "estimated_cost_usd": float} }
    """
    result = {}

    for record in log:
        model = record["model"]

        # TODO 1: If model not in result yet, initialise its sub-dict with zeros
        # Hint: use setdefault or check 'if model not in result'

        # TODO 2: Add this record's input_tokens and output_tokens to the running totals

        pass  # remove this when done

    # TODO 3: Compute estimated_cost_usd for each model
    # Formula: (total_input / 1_000_000) * costs[model]["input"]
    #        + (total_output / 1_000_000) * costs[model]["output"]
    for model, stats in result.items():
        stats["estimated_cost_usd"] = None  # replace with formula

    return result

import json
print(json.dumps(estimate_costs(call_log, COST_PER_M), indent=2))

**Questions to think about**
1. What happens if a record contains a model name that is not in `COST_PER_M`?
2. How would you extend this to also track the number of API calls per model?
3. Why is `d.setdefault(key, default)` better than `if key not in d: d[key] = default` in tight loops?

## Solution 1

In [ ]:
def estimate_costs(log: list, costs: dict) -> dict:
    result = {}

    for record in log:
        model = record["model"]

        # 1 — initialise if first time seeing this model
        if model not in result:
            result[model] = {"total_input_tokens": 0, "total_output_tokens": 0}

        # 2 — accumulate
        result[model]["total_input_tokens"]  += record["input_tokens"]
        result[model]["total_output_tokens"] += record["output_tokens"]

    # 3 — compute cost
    for model, stats in result.items():
        c = costs[model]
        stats["estimated_cost_usd"] = round(
            (stats["total_input_tokens"]  / 1_000_000) * c["input"] +
            (stats["total_output_tokens"] / 1_000_000) * c["output"],
            6
        )

    return result

print(json.dumps(estimate_costs(call_log, COST_PER_M), indent=2))

## Exercise 2 — Row-Oriented to Column-Oriented Transformation

**Goal**: Transform a list-of-dicts (row-oriented) into a dict-of-lists (column-oriented) that Pandas can consume directly.

**Why it matters**: APIs return row-oriented JSON. Pandas DataFrames are column-oriented. This transformation is one you will write in almost every data ingestion script.

In [ ]:
rows = [
    {"id": 1, "text": "كيف حالك؟", "label": "question",  "length": 9,  "confidence": 0.95},
    {"id": 2, "text": "أنا بخير",    "label": "statement", "length": 7,  "confidence": 0.88},
    {"id": 3, "text": "شكراً جزيلاً","label": "statement", "length": 11, "confidence": 0.91},
    {"id": 4, "text": "ما اسمك؟",   "label": "question",  "length": 8,  "confidence": 0.97},
]

def rows_to_cols(rows: list) -> dict:
    """
    Convert a list of dicts (row-oriented) to a dict of lists (column-oriented).
    The keys of the output dict are the column names.
    Works for any schema — do NOT hard-code column names.
    """
    # TODO: implement without hard-coding any column names
    # Hint: get the column names from rows[0].keys(), then build lists
    pass

cols = rows_to_cols(rows)
print("Column-oriented dict:")
for k, v in cols.items():
    print(f"  {k}: {v}")

# Verify it works with Pandas
import pandas as pd
df = pd.DataFrame(cols)
print("\nDataFrame:")
print(df)

## Solution 2

In [ ]:
def rows_to_cols(rows: list) -> dict:
    if not rows:
        return {}
    cols = {key: [] for key in rows[0].keys()}   # init empty lists
    for row in rows:
        for key, val in row.items():
            cols[key].append(val)
    return cols

cols = rows_to_cols(rows)
import pandas as pd
df = pd.DataFrame(cols)
print(df)

# One-liner alternative (works when all rows have same keys)
cols2 = {k: [r[k] for r in rows] for k in rows[0]}
print("\nOne-liner result equal:", cols == cols2)

## Common Mistakes

### Mistake 1 — `KeyError` vs `get()`

In [ ]:
record = {"model": "gpt-4o", "tokens": 512}

# WRONG — crashes if key missing
try:
    print(record["temperature"])
except KeyError as e:
    print(f"KeyError: {e}")

# CORRECT — provide a safe default
print(record.get("temperature", 0.7))   # 0.7

### Mistake 2 — Mutable default value shared across calls

In [ ]:
# WRONG — the default [] is created ONCE and shared
def add_token(token, history={}):
    history.setdefault("tokens", []).append(token)
    return history

print(add_token("hello"))
print(add_token("world"))   # history is the SAME object!

# CORRECT — use None as sentinel
def add_token_safe(token, history=None):
    if history is None:
        history = {}
    history.setdefault("tokens", []).append(token)
    return history

### Mistake 3 — Modifying a dict while iterating

In [ ]:
d = {"a": 1, "b": 2, "c": 3}

# WRONG — RuntimeError: dictionary changed size during iteration
try:
    for k in d:
        if d[k] < 2:
            del d[k]
except RuntimeError as e:
    print(f"RuntimeError: {e}")

# CORRECT — iterate over a copy of keys
for k in list(d.keys()):
    if d[k] < 2:
        del d[k]
print(d)  # {"b": 2, "c": 3}

## Real AI Engineering Usage

```python
# Config management — nested dicts for training hyperparameters
config = {"model": {...}, "training": {"lr": 2e-5, "epochs": 3}, "data": {...}}

# RAG metadata record — one dict per chunk
chunk = {"text": "...", "source": "paper.pdf", "page": 4,
         "language": "ar", "embedding": [0.1, 0.2, ...]}

# Token usage accumulator — dict of dicts
usage = {}
usage.setdefault(model, {"in": 0, "out": 0})["in"] += tokens_in

# Prompt template variables — format strings from dicts
template = "Summarise this in {lang}: {text}"
print(template.format(**{"lang": "Arabic", "text": "Hello world"}))

# Deduplication — use dict keys as a seen-set (O(1) lookup)
seen = {}
unique_docs = [seen.setdefault(d["id"], d) for d in docs if d["id"] not in seen]

# Vocabulary mapping — token-to-id dict in tokenisers
vocab = {"[PAD]": 0, "[UNK]": 1, "مرحبا": 2, "world": 3}
token_id = vocab.get(word, vocab["[UNK]"])
```

## Final Mini Exercise — Build a Simple Vocabulary Counter

Given a list of Arabic/English sentences, build:
1. A word-frequency dict `{word: count}`
2. A `top_n(freq_dict, n)` function that returns the n most common words as a dict
3. A token-to-id mapping `{word: id}` sorted by frequency (most common = lowest id)

In [ ]:
sentences = [
    "the cat sat on the mat",
    "the cat is on the mat",
    "a cat sat on a mat",
    "the mat is on the floor",
]

def build_vocab(sentences: list) -> tuple:
    """Returns (freq_dict, token_to_id)"""
    # TODO: implement
    pass

freq, tok2id = build_vocab(sentences)
print("Freq:", dict(list(freq.items())[:5]))
print("tok2id:", dict(list(tok2id.items())[:5]))

## Key Takeaways

- Dicts are **O(1) lookup** hash maps — the most-used data structure in Python AI code.
- `d.get(key, default)` is safer than `d[key]` in production code where keys may be absent.
- **Row-oriented** (list of dicts) is the JSON API format. **Column-oriented** (dict of lists) is the Pandas format. Know how to convert between them.
- `setdefault` and `defaultdict` eliminate boilerplate initialisation in accumulation patterns.
- Never modify a dict while iterating over it — iterate over `list(d.keys())` instead.
- Dict comprehensions `{k: f(v) for k, v in d.items()}` are idiomatic and fast.
- In Python 3.7+, dicts preserve insertion order — rely on it for pipeline stages.